# Soybean_2k: merge data and create ID-disjoint train/test sets

按性状进行全局 ID 级互斥划分，并尽量使各环境测试比例接近目标值。特征筛选、缺失值填补和标准化均只使用训练集信息。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

WORK_DIR = Path("/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/")
PHENOTYPE_FILE = WORK_DIR / "phen_matrix_soybean.txt"
HAPLOTYPE_FILE = WORK_DIR / "soybean_2k_hap_per.txt"
OUTPUT_DIR = WORK_DIR / "trainTestData"
ID_COL, ENV_COLS = "ID", ["location", "year"]
TRAITS = [
    "BBD","BPD","MD","FA16C","FA18C",
    "FAC","HSW","SL","ST","LL","LW",
    "NN","OB","PLH","VBN","PRT",
]


TEST_SIZE, RANDOM_STATE, N_CANDIDATES = 0.10, 42, 20000

In [2]:
phenotypes = pd.read_table(PHENOTYPE_FILE)
haplotypes = pd.read_table(HAPLOTYPE_FILE)

if haplotypes["sample"].duplicated().any():
    raise ValueError("单倍型频率表中的 sample 必须唯一")

haplotypes = haplotypes[haplotypes["sample"].isin(phenotypes[ID_COL].unique())]

df_merge = (phenotypes.merge(
    haplotypes, left_on=ID_COL, right_on="sample", how="left",
    validate="many_to_one").drop(columns="sample"))

print(f"Merged data: {df_merge.shape[0]:,} rows × {df_merge.shape[1]:,} columns")

Merged data: 8,600 rows × 234 columns


In [3]:
def make_balanced_id_split(df, id_col, env_cols, trait, test_size=0.10,
                           random_state=42, n_candidates=20000):
    if not 0 < test_size < 1:
        raise ValueError("test_size 必须在 0 和 1 之间")
    if not isinstance(n_candidates, int) or n_candidates < 1:
        raise ValueError("n_candidates 必须是正整数")

    required = [id_col, *env_cols, trait]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"缺少必要列：{missing}")

    basis = (df.loc[df[trait].notna(), [id_col, *env_cols]]
               .dropna().drop_duplicates().copy())
    basis["_environment"] = basis[env_cols].astype(str).agg("||".join, axis=1)
    presence = pd.crosstab(basis[id_col], basis["_environment"]).gt(0).astype(np.uint8)
    eligible_ids = presence.index.to_numpy()
    matrix = presence.to_numpy(dtype=np.uint8)
    n_ids = len(eligible_ids)
    if n_ids < 2:
        raise ValueError(f"{trait}: 有效 ID 少于 2 个，无法划分")

    n_test = min(max(1, round(n_ids * test_size)), n_ids - 1)
    totals = matrix.sum(axis=0)
    targets = np.rint(totals * test_size).astype(int)
    rng = np.random.default_rng(random_state)
    best_indices = best_score = None

    for _ in range(n_candidates):
        indices = rng.choice(n_ids, size=n_test, replace=False)
        counts = matrix[indices].sum(axis=0)
        if (counts == 0).any() or (counts >= totals).any():
            continue
        deviations = np.abs(counts / totals - test_size)
        score = (float(deviations.max()), float(deviations.mean()),
                 int(np.abs(counts - targets).sum()))
        if best_score is None or score < best_score:
            best_indices, best_score = indices.copy(), score

    if best_indices is None:
        raise RuntimeError(f"{trait}: 未找到使每个环境同时含 train/test 的 ID 划分")

    test_ids = set(eligible_ids[best_indices])
    manifest = pd.DataFrame({
        id_col: eligible_ids,
        "split": np.where(pd.Series(eligible_ids).isin(test_ids), "test", "train"),
    })
    test_counts = matrix[best_indices].sum(axis=0)
    summary = pd.DataFrame({
        "environment": presence.columns,
        "n_total": totals,
        "n_test": test_counts,
    })
    summary["n_train"] = summary["n_total"] - summary["n_test"]
    summary["test_ratio"] = summary["n_test"] / summary["n_total"]
    summary["deviation_from_target"] = (summary["test_ratio"] - test_size).abs()
    print(
        f"{trait}: eligible IDs={n_ids}, train={n_ids - len(test_ids)}, "
        f"test={len(test_ids)}, max environment deviation="
        f"{summary['deviation_from_target'].max():.4f}"
    )
    return test_ids, manifest, summary


def fill_from_training(train_df, test_df, env_cols, feature_cols):
    group_medians = train_df.groupby(env_cols, dropna=False)[feature_cols].median()
    global_medians = train_df[feature_cols].median()

    def fill(frame):
        result = frame.copy()
        keys = (result[env_cols[0]] if len(env_cols) == 1
                else pd.MultiIndex.from_frame(result[env_cols]))
        aligned = group_medians.reindex(keys)
        aligned.index = result.index
        result[feature_cols] = result[feature_cols].fillna(aligned).fillna(global_medians)
        return result

    return fill(train_df), fill(test_df)


def export_trait_split(df, trait, id_col, env_cols, test_ids, output_dir,
                       test_size=0.10):
    required = [id_col, *env_cols, trait]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{trait}: 缺少必要列：{missing}")

    base = df.dropna(subset=required).copy()
    key_cols = [id_col, *env_cols]
    duplicates = base.duplicated(key_cols, keep=False)
    if duplicates.any():
        example = base.loc[duplicates, key_cols].sort_values(key_cols).head(20)
        raise ValueError(f"{trait}: 存在重复 ID × environment 记录：\n{example}")

    is_test = base[id_col].isin(test_ids)
    train_base, test_base = base.loc[~is_test].copy(), base.loc[is_test].copy()
    train_ids, test_trait_ids = set(train_base[id_col]), set(test_base[id_col])
    if not train_ids or not test_trait_ids or not train_ids.isdisjoint(test_trait_ids):
        raise ValueError(f"{trait}: train/test ID 划分无效")

    counts = (pd.concat([
        train_base[env_cols].assign(split="train"),
        test_base[env_cols].assign(split="test"),
    ]).groupby([*env_cols, "split"]).size().unstack(fill_value=0)
      .reindex(columns=["train", "test"], fill_value=0).reset_index())
    if (counts[["train", "test"]] == 0).any().any():
        raise ValueError(f"{trait}: 某环境缺少 train 或 test")

    protected = {id_col, trait, *env_cols}
    candidates = [c for c in base.columns if c not in protected]
    retained = [c for c in candidates if train_base[c].isna().mean() <= 0.5]
    if not retained:
        raise ValueError(f"{trait}: 没有特征通过训练集缺失率 <= 0.5 的筛选")
    non_numeric = [c for c in retained if not pd.api.types.is_numeric_dtype(train_base[c])]
    if non_numeric:
        raise TypeError(f"{trait}: 非数值预测特征：{non_numeric}")

    retained_cols = [c for c in base.columns if c in protected or c in retained]
    rename_map = {c: f"fea_{c}" for c in retained}
    train_base = train_base[retained_cols].rename(columns=rename_map)
    test_base = test_base[retained_cols].rename(columns=rename_map)
    feature_cols = list(rename_map.values())
    output_dir.mkdir(parents=True, exist_ok=True)

    pd.concat([train_base, test_base]).sort_index().to_csv(
        output_dir / f"data-{trait}.txt", sep="\t", index=False)

    raw_scaler = StandardScaler().fit(train_base[feature_cols])
    train_cat, test_cat = train_base.copy(), test_base.copy()
    train_cat[feature_cols] = raw_scaler.transform(train_cat[feature_cols])
    test_cat[feature_cols] = raw_scaler.transform(test_cat[feature_cols])
    train_cat.to_csv(output_dir / f"train-{trait}.txt", sep="\t", index=False)
    test_cat.to_csv(output_dir / f"test-{trait}.txt", sep="\t", index=False)

    train_fill, test_fill = fill_from_training(
        train_base, test_base, env_cols, feature_cols)
    if train_fill[feature_cols].isna().any().any() or test_fill[feature_cols].isna().any().any():
        raise ValueError(f"{trait}: 训练统计量填补后仍有缺失值")
    fill_scaler = StandardScaler().fit(train_fill[feature_cols])
    train_fill[feature_cols] = fill_scaler.transform(train_fill[feature_cols])
    test_fill[feature_cols] = fill_scaler.transform(test_fill[feature_cols])
    train_fill.to_csv(output_dir / f"trainFill-{trait}.txt", sep="\t", index=False)
    test_fill.to_csv(output_dir / f"testFill-{trait}.txt", sep="\t", index=False)

    counts["n_total"] = counts["train"] + counts["test"]
    counts["test_ratio"] = counts["test"] / counts["n_total"]
    counts["deviation_from_target"] = (counts["test_ratio"] - test_size).abs()
    counts.insert(0, "trait", trait)
    return counts


def build_all_splits(df, traits, id_col, env_cols, output_dir,
                     test_size=0.10, random_state=42, n_candidates=20000):
    manifests, balances, counts = [], [], []
    for trait in traits:
        test_ids, manifest, balance = make_balanced_id_split(
            df, id_col, env_cols, trait, test_size, random_state, n_candidates)
        manifest.insert(0, "trait", trait)
        balance.insert(0, "trait", trait)
        manifests.append(manifest)
        balances.append(balance)
        counts.append(export_trait_split(
            df, trait, id_col, env_cols, test_ids, output_dir, test_size))

    outputs = {
        "id-split-manifest.tsv": pd.concat(manifests, ignore_index=True),
        "id-split-balance.tsv": pd.concat(balances, ignore_index=True),
        "environment-split-counts.tsv": pd.concat(counts, ignore_index=True),
    }
    for filename, table in outputs.items():
        table.to_csv(output_dir / filename, sep="\t", index=False)
    return outputs

In [4]:
results = build_all_splits(
    df_merge, TRAITS, ID_COL, ENV_COLS, OUTPUT_DIR,
    TEST_SIZE, RANDOM_STATE, N_CANDIDATES,
)
display(results["environment-split-counts.tsv"])

BBD: eligible IDs=1470, train=1323, test=147, max environment deviation=0.0017
BPD: eligible IDs=1169, train=1052, test=117, max environment deviation=0.0005
MD: eligible IDs=1453, train=1308, test=145, max environment deviation=0.0025
FA16C: eligible IDs=758, train=682, test=76, max environment deviation=0.0007
FA18C: eligible IDs=759, train=683, test=76, max environment deviation=0.0007
FAC: eligible IDs=759, train=683, test=76, max environment deviation=0.0007
HSW: eligible IDs=1438, train=1294, test=144, max environment deviation=0.0022
SL: eligible IDs=1424, train=1282, test=142, max environment deviation=0.0007
ST: eligible IDs=1424, train=1282, test=142, max environment deviation=0.0007
LL: eligible IDs=1393, train=1254, test=139, max environment deviation=0.0013
LW: eligible IDs=1412, train=1271, test=141, max environment deviation=0.0013
NN: eligible IDs=812, train=731, test=81, max environment deviation=0.0002
OB: eligible IDs=729, train=656, test=73, max environment deviatio

split,trait,location,year,train,test,n_total,test_ratio,deviation_from_target
0,BBD,BeiJing,2013,714,78,792,0.098485,0.001515
1,BBD,BeiJing,2014,730,81,811,0.099877,0.000123
2,BBD,HeNan,2014,907,100,1007,0.099305,0.000695
3,BBD,HeNan,2015,887,98,985,0.099492,0.000508
4,BBD,HeiLongJiang,2014,302,34,336,0.101190,0.001190
...,...,...,...,...,...,...,...,...
89,VBN,ShanXi_Wet,2014,527,58,585,0.099145,0.000855
90,VBN,ShanXi_Wet,2015,534,58,592,0.097973,0.002027
91,PRT,BeiJing,2013,645,72,717,0.100418,0.000418
92,PRT,HeNan,2014,547,61,608,0.100329,0.000329
